# OT-2: Mean-Line Turbine Design

> Author: Elias Aoubala

> Date: 22/12/2025

In [1]:
%load_ext autoreload

%autoreload 2

from turborocket.meanline.meanline_relations import TurbineStageDesign
from turborocket.fluids.fluids import IdealGas
import CoolProp.CoolProp as CP

import numpy as np
import pandas as pd

from datetime import datetime

import yaml

import handcalcs.render

## 1 - Background

This document encapsulates the mean-line design of the turbine-stage of the `OT-2: Man-Ray and the Dirty Bubble`. A high level mean line design study is conducted using our in-house python package `turboRocket`, from which an iterative optimisation is performed to evaluate for our key gas-generator mass flows needed to meet our power requirements.

## 2 - High Level Functional Parmaeters (as loaded from yaml file)

We have now refactored our code so the input files are stored in `yaml` files for both the gas-generator characteristics and the turbine user selected inputs.

Hence, we just load them in via yaml files.

In [2]:
input_directory = "config"

### 2.1 - Gas Generator Inputs

In [3]:
with open(f"{input_directory}/gg_out.yaml") as f:

    gg_inputs = yaml.safe_load(f)

We can extract our gas object properties accordingly.

In [4]:
gas_dic = gg_inputs["gas_generator"]["operating_point"]["gas"]

fluid = IdealGas(
    name="GG Exhaust",
    p=gas_dic["P"],
    t=gas_dic["T"],
    gamma=gas_dic["gamma"],
    cp=gas_dic["cp"],
    R=gas_dic["R"],
)

And we can extract our geometries for our nozzle throat and mass flow.

In [5]:
gg_m_dot = gg_inputs["gas_generator"]["operating_point"]["m_dot"]

gg_geom = gg_inputs["gas_generator"]["geometry"]["chamber"]

### 2.2 - Turbine Inputs

In [6]:
with open(f"{input_directory}/turbine_in.yaml") as f:

    turbine_input = yaml.safe_load(f)

We can extract our key parameters, in this case required power, stator parameters and rotor

In [7]:
turbine_rotor = turbine_input["turbine_stage"]["rotor"]
turbine_stator = turbine_input["turbine_stage"]["stator"]

turbine_power = turbine_input["turbine_stage"]["power"]

## 4 - Mean-Line Design Power Optimization

Our current approach for mean-line design does not design to user specification, but rather takes in some geometric parameters from the user, and calculates the expected turbine stage performance from it (in this case power, etc). As we have a target shaft power we must hit, this setions aims to conduct a primitive optimisation where the gas-generator mass flow is iterated on, until the target shaft power is met.

### 4.1 - Inital Parameter Guesses

As this problem is an optimisation problem, we will manually guess the blade spacing and thickness. These will updated once the turbine profile has been designed.

In [8]:
b = 15e-3
t = 5e-3

Based on these inputs, we can derive the required turbine power accordingly - using our refined loss model.

The main parameter we will iterate on the GG mass flow rate untill we reach our target power. We will do an adjoint-esque iterative study until the power requirement is reached.

We will instantiate an inital error of 0

In [9]:
error = 0

### 4.2 - Optimisation Loop for System Power

We will use a relaxation factor of **0.4** for the adjoint solver based on the relative error from the target power.

In [10]:
relax = 0.4

In [11]:
print(f"Blade Chord Length: {b*1e3}")
print(f"Blade Spacing: {t*1e3}")
print(turbine_rotor["N_shaft"])
m_dot_t = gg_m_dot["total"] 

Blade Chord Length: 15.0
Blade Spacing: 5.0
24000.0


In [12]:
m_dot_t -= m_dot_t*error*relax

stage = TurbineStageDesign(gas=fluid, m_dot=m_dot_t, omega=turbine_rotor["N_shaft"], alpha=(90 - turbine_stator["alpha"]))

stage.set_operating_point(u_cis=turbine_rotor["u_cis"], Rt=turbine_stator["Rt"], b=b, t=t, delta_r=turbine_rotor["delta_r"], N=turbine_stator["N_nozzle"])

result = stage.solve_performance(phi_n=turbine_stator["phi_n"])

P = result["performance"]["Power"]

error = (P - turbine_power) / turbine_power

print(f"Relative Error: {error*1e2:.2f} %")

Current Error: 38.89378527693544 %
Current Error: 2.7164761302485783 %
Current Error: 0.250212306399475 %
Current Error: 0.022536681227062847 %
Current Error: 0.0020340255013711283 %
20.0
beta_2: 0.38436448022445524
beta_1: 0.5205480276404707
Relative Error: -0.51 %


We can get out power produced and calculate our error to repeat once again.

**Final Gas Generator Mass Flow Rate**

In [13]:
%%render param

m= m_dot_t*1000

<IPython.core.display.Latex object>

## 5 - Expected Mean-Line Performance Metrics

We can now get an idea of the specific power of the turbine and the associated performance Metrics

In [14]:
performance_dic = result["performance"]
performance_dic = {k: [v] for k, v in performance_dic.items()}
performance_df = pd.DataFrame(performance_dic)

pressure_dic = result["pressure"]
pressure_dic = {k: [v] for k, v in pressure_dic.items()}
pressure_df = pd.DataFrame(pressure_dic)

velocity_dic = result["velocity"]
velocity_dic = {k: [v] for k, v in velocity_dic.items()}
velocity_df = pd.DataFrame(velocity_dic)

temperature_dic = result["temperature"]
temperature_dic = {k: [v] for k, v in temperature_dic.items()}
temperature_df = pd.DataFrame(temperature_dic)

geometry_dic = result["geometry"]
geometry_dic = {k: [v] for k, v in geometry_dic.items()}
geometry_df = pd.DataFrame(geometry_dic)

mach_dic = result["mach"]
mach_dic = {k: [v] for k, v in mach_dic.items()}
mach_df = pd.DataFrame(mach_dic)

angles_dic = result["angles"]
angles_dic = {k: [v] for k, v in angles_dic.items()}
angles_df = pd.DataFrame(angles_dic)

### 5.1 - High Level Functional Parameters

In [15]:
performance_df["Power (kW)"] = performance_df["Power"].multiply(1e-3)
performance_df["Torque (Nm)"] = performance_df["Power"] / (25000 * 2 *np.pi/60)

performance_df

,dh,eps,phi_r,phi_l,m_leakage,eta_l,phi,eta_h,zeta_eps,eta_o,Power,Power (kW),Torque (Nm)
0,1.214776e+06,0.042767,0.672626,0.198995,0.027859,0.801005,0.85,0.187385,0.009692,0.140405,23878.530863,23.878531,9.120927


As can be seen, we are getting an efficiency on the order of **45%** with a specific power of 987 kW/kg/s

This works out to a GG size of the following:

### 5.2 - Expected Stage Pressures

These are the expected stage pressures in **Bar**.

In [16]:
pressure_df * 1e-5

,p_0,p_1,p_1o,p_1o_r,p_2o_r,p_2o
0,25.000085,1.08696,61.214139,55.564019,20.658797,1.851081


### 5.3 - Expected Stage Temperatures

In [17]:
temperature_df

,t_0,t_1,t_1o_r,t_2,t_2o
0,970.165422,679.99798,951.565742,828.701421,921.75391


### 5.4 - Expected Stage Geometric Parameters

Here are the expected stage geometric parameters. We make a comparison between that calculated using the mean-line method, and that evaluated from our GG combustion solver to confirm the areas are comparable.

In [18]:
geometry_df["Nozzle Throat (mm)"] = [(gg_geom["a_t"] / (turbine_stator["N_nozzle"]* np.pi))**(1/2) * 2 * 1e3]
geometry_df["Nozzle Exit (mm)"] = geometry_df["s_c"]*1e3
geometry_df["eps"] = geometry_df["A_1"] / geometry_df["A_0"]

throat_error = (geometry_df["A_0"][0] - gg_geom["a_t"] )/ gg_geom["a_t"]

print(f"Comparison of Throat Calculated using GG combustion solver: {throat_error*100:.1f} %")

geometry_df

Comparison of Throat Calculated using GG combustion solver: -5.7 %


,D_m,A_1,A_0,s_c,s_b,D_hub,D_tip,AR,Nozzle Throat (mm),Nozzle Exit (mm),eps
0,0.09923,0.000355,0.000058,0.008683,0.011183,0.088047,0.110413,0.745545,3.611191,8.683178,6.132198


### 5.5 - Expected Mach Numbers at Each Stage

In [19]:
mach_df

,m_star_c1,m_star_w1,m_star_w2,m_star_c2
0,1.641177,1.603148,1.07832,0.876265


### 5.6 - Expected Angles

For these angles, we subtracting from 90 to get the relative to the axial direction.

In [20]:
angles_df

,beta_1,beta_2,alpha_2
0,22.022463,0.520548,47.911851


### 5.7 - Expected Stage Velocities

These are the expected absolute and relative velocities passing through the stage.

In [21]:
velocity_df

,u,c_1s,c_1,w_1,a_star_2,w_2,c_2,a_star_3
0,124.69616,1558.701995,1324.896696,1208.473432,753.812643,812.850903,707.395018,741.910491


## 6 - Spin Start Power Specifications

In this section, we evaluate for the spin start requirements for the turbine.

### 6.1 - High Level Requirements

The High Level Requirements of the system are as follows:

In [22]:
dp_nom = 40e5 # Pa
n_nom = 24e3 # rpm
w_nom = turbine_power

dp_target = 10e5 # Pa 

We can then use the affinity laws to evaluate for our expected shaft speed and power to achieve this requirement

In [23]:
n_spin = n_nom * (dp_target/dp_nom)**(1/2)

w_spin = (n_spin/n_nom)**(3) * w_nom

print(f"Spin-up Shaft Speed: {n_spin/1e3:.2f}k RPM")
print(f"Spin-up Shaft Power: {w_spin/1e3:.2f} kW")

Spin-up Shaft Speed: 12.00k RPM
Spin-up Shaft Power: 3.00 kW


We can finally specify the requiremetns of the spin-up machine.

In [24]:
eta_spin = 0.1 # Gas Efficency
t_spin = 273.15 + 15 # Gas Stagnation Temperature
p_spin = 80e5 # Gas Stangation Pressure
p_amb = 1e5 # Ambient Pressure
V_l = 20e-3 # Tank Volume
gas = "Nitrogen"

### 6.2 - Spin Start Mass-Flow Requirement

We will use cool-prop and employ the use of the ideal gas assumption to derive our gas properties.

In [25]:
cp = CP.PropsSI("C", "P", p_spin, "T", t_spin, gas)
gamma = CP.PropsSI("C", "P", p_spin, "T", t_spin, gas)/CP.PropsSI("O", "P", p_spin, "T", t_spin, gas)
rho = CP.PropsSI("D", "P", p_spin, "T", t_spin, gas)
R = CP.PropsSI("GAS_CONSTANT", gas)/CP.PropsSI("M", gas)

dh = cp * t_spin * (1 - (p_amb / p_spin) ** ((gamma - 1) / gamma))

m_dot = w_spin / (eta_spin * dh)
print(f"Required Nitrous Mass Flow: {m_dot*1e3:.2f} g/s")

Required Nitrous Mass Flow: 111.86 g/s


### 6.3 - Spin Start Nozzle Sizing

We can now evaluate for the required throat area and exit area of the spin-start nozzle accordingly.

In [26]:
A_t = m_dot / (gamma * rho * p_spin * (2/(gamma + 1))**((gamma + 1)/(gamma - 1)))**(1/2)

d_spin = 2 * (A_t/np.pi)**(1/2)

print(f"Spin Start Throat Diameter: {d_spin*1e3:.2f} mm")

Spin Start Throat Diameter: 2.71 mm


We can then find the exit area for an adiabatic flow pretty simply

$$ h_o = h + \frac{V^2}{2}

$$ V = \sqrt{2(h_o - h)} = \sqrt{2 * dh}

$$ \dot{m} = \rho A V $$

$$ \dot{m} = \frac{P}{R T} A V $$

$$ A = \frac{\dot{m} R T}{P V}

$$ h = C_p T $$

$$ A = \frac{\dot{m} R h}{P V C_p}

From there, we can solve for our conditons

In [27]:
v_spin = (2 * dh )**(1/2)

h = cp*t_spin - dh

a_exit_spin = (m_dot * h * R)/(p_amb * v_spin * cp)

d_exit_spin = (a_exit_spin / np.pi)**(1/2) * 2

print(f"Exit Diameter: {d_exit_spin*1e3:.2f} mm")

Exit Diameter: 5.94 mm


## 6 - Exporting our Results

Now that we have done the high level mean-line design, we can extract our key parameters derived from the study, that we can place in a data fram and export as a yaml file, that can be read.

In [28]:
performance_out = { "performance": {k: float(v[0]) for k, v in performance_df.items()} }

pressure_out = { "pressure": {k: float(v[0]) for k, v in pressure_df.items()} }

velocity_out = { "velocity": {k: float(v[0]) for k, v in velocity_df.items()} }

temperature_out = { "temperature": {k: float(v[0]) for k, v in temperature_df.items()} }

geometry_out = { "geometry": {k: float(v[0]) for k, v in geometry_df.items()} }

mach_out = { "mach": {k: float(v[0]) for k, v in mach_df.items()} }

angles_out = { "angles": {k: float(v[0]) for k, v in angles_df.items()} }

mass_flow_out = {"error": {"input_val": float(gg_m_dot["total"]), "solved": float(m_dot_t), "m_dot_error": float((m_dot_t - gg_m_dot["total"])*100/gg_m_dot["total"]), "power_error": float(error)*100}}

input_params = {"input": turbine_input["turbine_stage"]}


We can now save this output file accordingly.

In [29]:
with open(f"{input_directory}/turbine_out.yaml", "w") as f:
    
    # Hight Level Title
    f.write(f"#"*80 + "\n")
    f.write(f"#{"OT-2: Man-Ray and The Dirty Bubble":^78}#\n")
    f.write(f"#{f"{"":-^40}":^78}#\n")
    f.write(f"#{"Meanline Design Study Results":^78}#\n")
    f.write(f"#{"":^78}#\n")
    f.write(f"#{f"Generated at: XX:{datetime.now().minute}"::^78}#\n")
    f.write(f"#{"":^78}#\n")
    f.write(f"#"*80 + "\n")
    f.write(f"\n\n")
    
    # Solution Convergence
    f.write(f"#"*80 + "\n")
    f.write(f"#{"Convergence Errors on Mass Flows":-^78}#\n")
    f.write(f"#"*80 + "\n")
    yaml.dump(mass_flow_out, f, default_flow_style=False)
    f.write(f"\n")
    
    # Input Parameters
    f.write(f"#"*80 + "\n")
    f.write(f"#{" Input Parameters ":-^78}#\n")
    f.write(f"#"*80 + "\n")
    yaml.dump(input_params, f, default_flow_style=False)
    f.write(f"\n")
    
    
    # Performance Results
    f.write(f"#"*80 + "\n")
    f.write(f"#{" Performance Results ":-^78}#\n")
    f.write(f"#"*80 + "\n")
    yaml.dump(performance_out, f, default_flow_style=False)
    f.write(f"\n")
    
    # Pressure Results
    f.write(f"#"*80 + "\n")
    f.write(f"#{" Pressure Results ":-^78}#\n")
    f.write(f"#"*80 + "\n")
    yaml.dump(pressure_out, f, default_flow_style=False)
    f.write(f"\n")
    
    # Velocity Results
    f.write(f"#"*80 + "\n")
    f.write(f"#{" Velocity Results ":-^78}#\n")
    f.write(f"#"*80 + "\n")
    yaml.dump(velocity_out, f, default_flow_style=False)
    f.write(f"\n")
    
    # Temperature Results
    f.write(f"#"*80 + "\n")
    f.write(f"#{" Temperature Results ":-^78}#\n")
    f.write(f"#"*80 + "\n")
    yaml.dump(temperature_out, f, default_flow_style=False)
    f.write(f"\n")
    
    # Geometry Results
    f.write(f"#"*80 + "\n")
    f.write(f"#{" Geometry Results ":-^78}#\n")
    f.write(f"#"*80 + "\n")
    yaml.dump(geometry_out, f, default_flow_style=False)
    f.write(f"\n")
    
    # Mach Results
    f.write(f"#"*80 + "\n")
    f.write(f"#{" Mach Results ":-^78}#\n")
    f.write(f"#"*80 + "\n")
    yaml.dump(mach_out, f, default_flow_style=False)
    f.write(f"\n")
    
    # Angles Results
    f.write(f"#"*80 + "\n")
    f.write(f"#{" Angles Results ":-^78}#\n")
    f.write(f"#"*80 + "\n")
    yaml.dump(angles_out, f, default_flow_style=False)
    f.write(f"\n")
    


##